In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask.dataframe as dd
import pickle
import os

In [2]:
# —Later— to load:
with open('./estimators.pkl', 'rb') as f:
    loaded = pickle.load(f)
    combined_means = loaded['means']
    combined_stds  = loaded['stds']

In [4]:

# —Later— to reload:
with open('all_maximas_summary.pkl', 'rb') as f:
    summary_stats = pickle.load(f)


In [ ]:
# Extract data from summary_stats for plotting
summary_df = pd.DataFrame({series: stats.loc['mean'] for series, stats in summary_stats.items()})

# Transpose the DataFrame to have series types as columns and calculation methods as rows
summary_df = summary_df.T

# Plot grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(summary_df.index))  # Series types
width = 0.25  # Width of each bar

for i, col in enumerate(summary_df.columns):  # Iterate over calculation methods
    ax.bar(x + i * width, summary_df[col], width, label=col)

# Add labels, title, and legend
ax.set_xticks(x + width)
ax.set_xticklabels(summary_df.index, rotation=45)
ax.set_ylabel('Mean Value')
ax.set_title('Summary Statistics: Mean Values by Series and Calculation Method')
ax.legend(title='Calculation Method')

plt.tight_layout()
plt.show()

In [12]:
for i, (name, means) in enumerate(combined_means.items()):
   for series_name in means.keys():
        combined_means[name][series_name] = means[series_name].rename(
            lambda x: x.replace('_mean', ''), axis=1)

for i, (name, stds) in enumerate(combined_stds.items()):
   for series_name in stds.keys():
        combined_stds[name][series_name] = stds[series_name].rename(
            lambda x: x.replace('_std', ''), axis=1)

In [ ]:
# Create figures for means and standard deviations
figures = {}
axes = {}

for metric, combined_data, x_limits, title in [
    ('means', combined_means, (2.5, 4.5), 'Distributions of Means'),
    ('stds', combined_stds, (0, 1), 'Distributions of Standard Deviations')
]:
    series_names = combined_data[list(combined_data.keys())[0]].keys()
    series_names = sorted(series_names)
    fig, axs = plt.subplots(len(series_names), 1, figsize=(12, 8), constrained_layout=True)
    figures[metric] = fig
    axes[metric] = axs

    for row_idx, name in enumerate(combined_data.keys()):
        data = combined_data[name]

        for col_idx, series_name in enumerate(series_names):
            ax = axs[col_idx]
            ax.set_xlim(*x_limits)  # Adjust x-axis limits for better visibility
            true_stats = summary_stats[series_name]

            for col in data[series_name].columns:
                values = data[series_name][col].values
                true_value = true_stats[col]['mean'] if metric == 'means' else true_stats[col]['std']

                bins = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 121)  # 30 bins across x limits
                ax.hist(values, bins=bins, alpha=0.1, label=f'{col} {name}')
                ax.axvline(true_value, color='r' if metric == 'means' else 'b', linestyle='--', linewidth=2, label=None)
                # print(f"{series_name} {col} {name} std: {np.std(values)}")
                # q1, q3 = np.percentile(values, [25, 75])
                # print(f"{series_name} {col} {name} middle 50th percentile range: {q3 - q1}")
                # print()

            ax.set_title(f'{series_name}')
            ax.set_xlabel('Estimate')
            ax.set_ylabel('Count')

            if col_idx == len(series_names) - 1:
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')  # Move legend outside to the right

    fig.suptitle(title, fontsize=16)
    plt.show()


In [ ]:
# Assuming we already have:
# 1) combined_means: Dict of dataframes with mean estimates
# 2) combined_stds: Dict of dataframes with std estimates
# 3) summary_stats: Dict with true statistics from the dask compute

# --- Prepare result DataFrames ---
# First identify which estimator types we have
estimator_types = list(combined_means.keys())
# For each estimator type, identify which series are available
series_names = set()
for est_type in estimator_types:
    series_names.update(combined_means[est_type].keys())
series_names = sorted(list(series_names))  # Ensure consistent ordering

# Create a function to process a specific estimator type
def compute_metrics(est_type, combined_df):
    # Identify columns based on first available series/estimator
    first_series = next(iter(combined_df[est_type].keys()))
    cols = combined_df[est_type][first_series].columns
    
    bias = pd.DataFrame(index=series_names, columns=cols, dtype=float)
    se = pd.DataFrame(index=series_names, columns=cols, dtype=float)
    mse = pd.DataFrame(index=series_names, columns=cols, dtype=float)
    iqr = pd.DataFrame(index=series_names, columns=cols, dtype=float)
    
    # --- Fill with metrics ---
    for name in series_names:
        if name in combined_df[est_type]:
            for col in cols:
                est = combined_df[est_type][name][col].dropna().values
                true = summary_stats[name][col]['mean']
                # true = summary_stats[name]["SBM"]['mean']
                
                bias.at[name, col] = est.mean() - true
                se.at[name, col] = est.std(ddof=0)
                mse.at[name, col] = np.mean((est - true) ** 2)
                iqr.at[name, col] = np.percentile(est, 75) - np.percentile(est, 25)

                if np.isnan(bias.at[name, col]):
                    pass
                
    return {
        'bias': bias, 'se': se, 'mse': mse, 'iqr': iqr,
    }

# Compute metrics for each estimator type
results_means = {}
results_stds = {}
for est_type in estimator_types:
    std_est_type = est_type.replace('_estimator', '_std_estimator')
    results_means[est_type] = compute_metrics(est_type, combined_means)
    results_stds[est_type] = compute_metrics(std_est_type, combined_stds)
    
    print(f"\n=== Metrics for {est_type} ===")
    print("Bias of mean estimates:\n",  results_means[est_type]['bias'])
    print("\nSE of mean estimates:\n",  results_means[est_type]['se'])
    print("\nMSE of mean estimates:\n", results_means[est_type]['mse'])
    print("\nIQR of mean estimates:\n", results_means[est_type]['iqr'])
    
    print("\nBias of std estimates:\n", results_stds[est_type]['bias'])
    print("\nSE of std estimates:\n",   results_stds[est_type]['se'])
    print("\nMSE of std estimates:\n",  results_stds[est_type]['mse'])
    print("\nIQR of std estimates:\n",  results_stds[est_type]['iqr'])


In [ ]:
titles = ["Bias of Mean Estimates", "SE of Mean Estimates", "MSE of Mean Estimates", "IQR of Mean Estimates", "Bias of Std Estimates", "SE of Std Estimates", "MSE of Std Estimates", "IQR of Std Estimates"]
metrics = ['bias', 'se', 'mse', 'iqr']

# Transpose the DataFrame to swap columns and rows
for fig_idx, (results, metric_titles) in enumerate(zip([results_means, results_stds], 
                                                       [titles[:len(metrics)], titles[len(metrics):]])):
    fig, axs = plt.subplots(len(metrics), len(estimator_types), figsize=(15, 5 * len(metrics)), layout='tight', sharey='row')  # Share y-axis across rows

    for col_idx, est_type in enumerate(estimator_types):  # Iterate over columns (estimator types)
        for row_idx, (metric, title) in enumerate(zip(metrics, metric_titles)):  # Iterate over rows (metrics)
            ax = axs[row_idx, col_idx] if len(estimator_types) > 1 else axs[row_idx]
            
            # Get the metric dataframe for this estimator and transpose it
            df = results[est_type][metric].T
            
            # For each series, plot a grouped bar chart
            x = np.arange(len(df.index))
            width = 0.7 / len(df.columns)  # Width of each bar
            
            for j, col in enumerate(df.columns):
                offset = width * j - (len(df.columns) - 1) * width / 2
                ax.bar(x + offset, df[col], width, alpha=0.5, 
                       label=f'{col}')
            
            ax.set_title(est_type.replace('_estimator', ''))
            ax.set_xticks(x)
            ax.set_xticklabels(df.index, rotation=45)  # Use transposed index as labels
            
            if row_idx == 0 and col_idx == len(estimator_types)-1:  # Only show legend on the first plot
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

            if col_idx == 0:  # Only set y-labels for the first column
                ax.set_ylabel(title)

    plt.tight_layout()
    plt.suptitle(f'Figure {fig_idx + 1}: {metric_titles[0].split()[2]} Metrics', fontsize=16, y=1.02)
    plt.show()
